In [5]:
import math

In [1]:
# Due cluster evidenti: uno con valori bassi (classe 'A'), uno con valori alti (classe 'B')
X_train = [[1.0, 2.0], [1.5, 1.8], [8.0, 8.0], [9.0, 8.5]]
y_train = ['A', 'A', 'B', 'B']

# Il punto da prevedere è palesemente vicino al cluster della classe 'A'
x_test = [2.0, 2.0]
k = 3

In [2]:
def euclidean_distance(p1, p2):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(p1, p2)))

In [ ]:
def knn(X_train, y_train, x_test, k):
    distances = []
    for x, y in zip(X_train, y_train):
        dist = euclidean_distance(x, x_test)
        distances.append((dist, y))
    distances.sort(key=lambda x: x[0])
    neighbors = distances[:k]
    classes = [label for _, label in neighbors]
    predicted_class = max(set(classes), key=classes.count)
    return predicted_class
        

In [6]:
res = knn(X_train, y_train, x_test, k)
print(f"Predicted class for {x_test}: {res}")

Predicted class for [2.0, 2.0]: A


In [7]:
import heapq

In [ ]:
def knn_optimized(point, X_train, y_train, k):
    max_heap = []
    
    for i in range(len(X_train)):
        dist = euclidean_distance(point, X_train[i])
        
        # Trucco del Max-Heap: salviamo la distanza negativa (-dist)
        if len(max_heap) < k:
            heapq.heappush(max_heap, (-dist, y_train[i]))
        else:
            # max_heap[0][0] è la distanza negativa maggiore (cioè la distanza reale più lunga tra i K attuali)
            if -dist > max_heap[0][0]: 
                # heappushpop prima inserisce il nuovo e poi rimuove il più piccolo (cioè il più grande in assoluto)
                heapq.heappushpop(max_heap, (-dist, y_train[i]))
                
    # Estraiamo le etichette. Non ci interessa l'ordine per votare.
    k_nearest_labels = [label for neg_dist, label in max_heap]
    
    return max(set(k_nearest_labels), key=k_nearest_labels.count)

In [ ]:
def knn_optimized(X_train, y_train, x_test, k):
    distances = []
    for x, y in zip(X_train, y_train):
        dist = euclidean_distance(x, x_test)
        distances.append((dist, y))
    neighbors = heapq.nsmallest(k, distances, key=lambda x: x[0])
    classes = [label for _, label in neighbors]
    predicted_class = max(set(classes), key=classes.count)
    return predicted_class

In [ ]:
def knn_optimized(X_train, y_train, x_test, k):
    # Passiamo un generatore direttamente a nsmallest. 
    # Le parentesi tonde interne creano il generatore, NON una lista.
    # Memoria utilizzata: O(K) invece di O(N)
    neighbors = heapq.nsmallest(
        k, 
        ((euclidean_distance(x, x_test), y) for x, y in zip(X_train, y_train)), 
        key=lambda item: item[0]
    )
    
    classes = [label for _, label in neighbors]
    return max(set(classes), key=classes.count)